In [ ]:
# TITULO: Entrenamiento Comparativo - Detector de Placas
import os
from ultralytics import YOLO
from datetime import datetime
import pandas as pd
import matplotlib.pyplot as plt

# Rutas de los datasets generados
YAML_CLEAN = '../../datasets/02_placas/data.yaml'
# YAML_BASELINE = '../../datasets/02_placas_baseline/data.yaml'

# Ruta de salida de modelos
MODELS_DIR = '../../models/02_placas'

# Fecha para versionado
DATE_STR = datetime.now().strftime('%Y%m%d')

print("Configuracion lista.")

Configuracion lista.


In [7]:
# Función para obtener el optimizador real cuando se usa 'auto'
def obtener_optimizador_real(modelo):
    """
    Recupera el nombre real del optimizador cuando la configuración es 'auto'.
    """
    try:
        # 1. Si el entrenamiento acaba de terminar y el objeto sigue en memoria RAM
        if hasattr(modelo, 'trainer') and modelo.trainer and hasattr(modelo.trainer, 'optimizer'):
            # El optimizador es un objeto (ej. <torch.optim.sgd.SGD object at 0x...>)
            # Obtenemos su nombre de clase real
            opt_obj = modelo.trainer.optimizer
            nombre_real = type(opt_obj).__name__
            
            # También podemos sacar el Learning Rate real final
            lr_final = opt_obj.param_groups[0]['lr']
            
            print(f"Decisión de 'Auto':")
            print(f"   • Optimizador:   {nombre_real}") # Dirá 'SGD' o 'AdamW'
            print(f"   • Learning Rate: {lr_final:.6f}")
            return

        # 2. Si el modelo fue cargado desde disco (.pt) y no hay trainer en memoria
        # Buscamos en los metadatos internos del archivo
        if hasattr(modelo, 'ckpt') and modelo.ckpt:
            train_args = modelo.ckpt.get('train_args', {})
            # A veces aquí también dice 'auto', en cuyo caso la única verdad está en los logs de texto
            print(f"Configuración guardada: {train_args.get('optimizer', 'Desconocido')}")
            print("Si aquí dice 'auto', por favor revisa el archivo '/runs/.../train/main.log'")

    except Exception as e:
        print(f"No se pudo recuperar automáticamente: {e}")

In [8]:
# Entrenamiento YOLOv11n con dataset Resplit
run_name_v11n_resplit_tl = f"100_v11n_resplit_tl"

print(f"Iniciando entrenamiento: {run_name_v11n_resplit_tl}")

# Cargar modelo Nano pre-entrenado
model_v11n_resplit_tl = YOLO('yolo11n.pt')

results_v11n_resplit_tl = model_v11n_resplit_tl.train(
    data=YAML_CLEAN,  # Dataset resplit
    model='yolo11n.pt',  # Modelo pre-entrenado
    project=MODELS_DIR,
    name=run_name_v11n_resplit_tl,

    epochs=300,            # Ajustable
    patience=50,          # Early stopping    
    batch=-1,          # Ajustable
    imgsz=1280,            # Tamaño de imagen    
    
    exist_ok=False,         # Sobrescribir si existe
    pretrained=True,
    optimizer='auto',
    verbose=True
)

Iniciando entrenamiento: 100_v11n_resplit_tl
Ultralytics 8.4.116 🚀 Python-3.10.20 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4070 Ti, 11873MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../../datasets/02_placas/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=300, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1280, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.93

/home/robertoplr/miniconda3/envs/proyecto/lib/python3.10/site-packages/ultralytics/utils/tal.py:401: UserWarning: cumsum_cuda_kernel does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at ../aten/src/ATen/Context.cpp:91.)
  sx = feats[0].new_full((w,), 1, dtype=dtype).cumsum(0) - (1 - grid_cell_offset)  # shift x
/home/robertoplr/miniconda3/envs/proyecto/lib/python3.10/site-packages/ultralytics/utils/tal.py:402: UserWarning: cumsum_cuda_kernel does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at ../aten/src/ATen/Context.cpp:91.)
  sy = feat

AutoBatch: Computing optimal batch size for imgsz=1280 at 60.0% GPU memory utilization.
AutoBatch: CUDA:0 (NVIDIA GeForce RTX 4070 Ti) 11.59G total, 3.44G reserved, 0.33G allocated, 7.83G free
      Params      GFLOPs  GPU_mem (GB)  forward (ms) backward (ms)                   input                  output
     2590035       25.75         1.762            17           nan      (1, 3, 1280, 1280)                    list
     2590035       51.51         3.525         21.16           nan      (2, 3, 1280, 1280)                    list
     2590035         103         6.025         22.91           nan      (4, 3, 1280, 1280)                    list
     2590035         206        11.325         37.47           nan      (8, 3, 1280, 1280)                    list
CUDA out of memory. Tried to allocate 50.00 MiB. GPU 0 has a total capacity of 11.59 GiB of which 57.31 MiB is free. Process 2496 has 71.29 MiB memory in use. Including non-PyTorch memory, this process has 10.70 GiB memory in use. O

/home/robertoplr/miniconda3/envs/proyecto/lib/python3.10/site-packages/ultralytics/utils/tal.py:401: UserWarning: cumsum_cuda_kernel does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at ../aten/src/ATen/Context.cpp:91.)
  sx = feats[0].new_full((w,), 1, dtype=dtype).cumsum(0) - (1 - grid_cell_offset)  # shift x
/home/robertoplr/miniconda3/envs/proyecto/lib/python3.10/site-packages/ultralytics/utils/tal.py:402: UserWarning: cumsum_cuda_kernel does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at ../aten/src/ATen/Context.cpp:91.)
  sy = feat

      1/300      2.02G      1.096       4.72      1.041          1       1280: 100% ━━━━━━━━━━━━ 265/265 9.9it/s 26.8s<0.1ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 17/17 12.1it/s 1.4s0.1s
                   all         99        115      0.969      0.939      0.981      0.744

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      2/300      2.02G      1.028      2.801     0.9268          7       1280: 0% ──────────── 0/265  0.1s

...

    298/300      2.09G     0.3019     0.1954     0.7801          4       1280: 100% ━━━━━━━━━━━━ 265/265 13.6it/s 19.4s0.1s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 17/17 19.6it/s 0.9s0.1s
                   all         99        115      0.999      0.991      0.995      0.927

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
    299/300      2.09G      0.286     0.1885     0.7821          4       1280: 0% ──────────── 1/265 1.9it/s 0.2s<2:23

/home/robertoplr/miniconda3/envs/proyecto/lib/python3.10/site-packages/ultralytics/utils/tal.py:401: UserWarning: cumsum_cuda_kernel does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at ../aten/src/ATen/Context.cpp:91.)
  sx = feats[0].new_full((w,), 1, dtype=dtype).cumsum(0) - (1 - grid_cell_offset)  # shift x
/home/robertoplr/miniconda3/envs/proyecto/lib/python3.10/site-packages/ultralytics/utils/tal.py:402: UserWarning: cumsum_cuda_kernel does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at ../aten/src/ATen/Context.cpp:91.)
  sy = feat

    299/300      2.09G     0.2983     0.1907     0.7851          2       1280: 100% ━━━━━━━━━━━━ 265/265 14.2it/s 18.7s0.1s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 17/17 21.8it/s 0.8s0.1s
                   all         99        115      0.999      0.991      0.995      0.925

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
    300/300      2.09G      0.364     0.2425     0.7503          4       1280: 0% ──────────── 0/265  0.2s

/home/robertoplr/miniconda3/envs/proyecto/lib/python3.10/site-packages/ultralytics/utils/tal.py:401: UserWarning: cumsum_cuda_kernel does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at ../aten/src/ATen/Context.cpp:91.)
  sx = feats[0].new_full((w,), 1, dtype=dtype).cumsum(0) - (1 - grid_cell_offset)  # shift x
/home/robertoplr/miniconda3/envs/proyecto/lib/python3.10/site-packages/ultralytics/utils/tal.py:402: UserWarning: cumsum_cuda_kernel does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at ../aten/src/ATen/Context.cpp:91.)
  sy = feat

    300/300      2.09G     0.2966     0.1907     0.7836          2       1280: 100% ━━━━━━━━━━━━ 265/265 13.8it/s 19.2s0.2s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 17/17 17.9it/s 0.9s0.1s
                   all         99        115      0.999      0.991      0.995      0.926

300 epochs completed in 1.835 hours.
Optimizer stripped from /home/robertoplr/Documentos/moca_proyecto/models/02_placas/100_v11n_resplit_tl/weights/last.pt, 5.7MB
Optimizer stripped from /home/robertoplr/Documentos/moca_proyecto/models/02_placas/100_v11n_resplit_tl/weights/best.pt, 5.7MB

Validating /home/robertoplr/Documentos/moca_proyecto/models/02_placas/100_v11n_resplit_tl/weights/best.pt...
Ultralytics 8.4.116 🚀 Python-3.10.20 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4070 Ti, 11873MiB)
YOLO11n summary (fused): 101 layers, 2,582,347 parameters, 0 gradients, 6.4 GFLOPs
                 Class     Images  Instances      Box(P          R     

In [9]:
# --- RECUPERAR HIPERPARÁMETROS REALES ---
# Ejecutar al finalizar el entrenamiento
print(f"Batch Size: {model_v11n_resplit_tl.trainer.args.batch}") 
print(f"Optimizer: {obtener_optimizador_real(model_v11n_resplit_tl)}")
print(f"Learning Rate inicial: {model_v11n_resplit_tl.trainer.args.lr0}")

Batch Size: 3
Decisión de 'Auto':
   • Optimizador:   AdamW
   • Learning Rate: 0.000027
Optimizer: None
Learning Rate inicial: 0.01


In [ ]:
print("Validando modelo 100_v11n_resplit_tl en split='test'...")

PROJECT_DIR = '../../../../models/02_placas'
run_name = f"100_v11n_resplit_tl"

# Cargar el MEJOR modelo resultante del entrenamiento anterior
model_yolov8n_baseline_tl = os.path.join(PROJECT_DIR, run_name, 'weights', 'best.pt')
best_model = YOLO(model_yolov8n_baseline_tl)

# Ejecutar validación en split='test'
metrics = best_model.val(
    split='test', 
    project=PROJECT_DIR, 
    name=f"{run_name}_eval", 
    imgsz=640,      # Tamaño de imagen
    batch=49,        # Mismo batch que entrenamiento
    plots=True       # Generar gráficos de métricas
)

print(f"\nResultados Finales en Test del dataset Baseline:")
print(f"   mAP@50:    {metrics.box.map50:.4f} (Precisión holgada)")
print(f"   mAP@50-95: {metrics.box.map:.4f}  (Precisión estricta )")

Validando modelo 100_v11n_resplit_tl en split='test'...
Ultralytics 8.4.116 🚀 Python-3.10.20 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4070 Ti, 11873MiB)
YOLO11n summary (fused): 101 layers, 2,582,347 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 8227.3±1335.2 MB/s, size: 1821.6 KB)
val: Scanning /home/robertoplr/Documentos/moca_proyecto/datasets/02_placas/test/labels... 100 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 100/100 3.2Kit/s 0.0s
val: New cache created: /home/robertoplr/Documentos/moca_proyecto/datasets/02_placas/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 3/3 2.0s/it 5.9s1.3s3s
                   all        100        123      0.983      0.967      0.974       0.85
Speed: 2.9ms preprocess, 1.7ms inference, 0.0ms loss, 0.3ms postprocess per image
Results saved to /home/robertoplr/Documentos/moca_proyecto/notebooks/02_placas/models/02_placas/100_

In [ ]:
import glob
import random

run_name = f"100_v11n_resplit_tl_inference"

# Tomar una imagen de prueba aleatoria
test_images = glob.glob('../../datasets/02_placas/test/images/*.jpg')
if test_images:
    sample_img = random.choice(test_images)
    
    # Prediccion
    res = model_v11n_resplit_tl.predict(sample_img, save=True, project=MODELS_DIR, name=run_name)
     
    print(f"Inferencia guardada en {res[0].save_dir}")
else:
    print("No se encontraron imagenes de prueba.")


image 1/1 /home/robertoplr/Documentos/moca_proyecto/notebooks/02_placas/../../datasets/02_placas/test/images/01768.jpg: 864x1280 1 license_plate, 4.9ms
Speed: 6.1ms preprocess, 4.9ms inference, 0.7ms postprocess per image at shape (1, 3, 864, 1280)
Results saved to /home/robertoplr/Documentos/moca_proyecto/models/02_placas/20260808_v11n_resplit_tl_inference
Inferencia guardada en /home/robertoplr/Documentos/moca_proyecto/models/02_placas/20260808_v11n_resplit_tl_inference


In [ ]:
# MODELS_DIR = '../../models/02_placas'
# Cargar mejores pesos
# Asegúrate que estos nombres coincidan exactamente con los definidos en las celdas de entrenamiento
run_name_v11 = f"100_v11n_resplit_tl"

path_v11_weights = os.path.join(MODELS_DIR, run_name_v11, 'weights', 'best.pt')

model_final_v11 = YOLO(path_v11_weights)

print("--- VALIDACION CRUZADA ---")

# 1. Validar Modelo YOLOv11n
metrics_v11 = model_final_v11.val(
    data=YAML_CLEAN, 
    split='test', 
    project=MODELS_DIR,
    name=f"{run_name_v11}_val"
)

print("\nRESULTADOS COMPARATIVOS (mAP50-95):")
print(f"Modelo YOLOv11n: {metrics_v11.box.map:.4f}")

--- VALIDACION CRUZADA ---
Ultralytics 8.4.116 🚀 Python-3.10.20 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4070 Ti, 11873MiB)
YOLO11n summary (fused): 101 layers, 2,582,347 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 10365.6±1889.9 MB/s, size: 2368.4 KB)
val: Scanning /home/robertoplr/Documentos/moca_proyecto/datasets/02_placas/test/labels.cache... 100 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 100/100 20.0Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 7/7 1.8it/s 4.0s0.2ss
                   all        100        123          1      0.983      0.995       0.92
Speed: 7.9ms preprocess, 4.8ms inference, 0.0ms loss, 0.6ms postprocess per image
Results saved to /home/robertoplr/Documentos/moca_proyecto/notebooks/02_placas/models/02_placas/100_v11n_resplit_tl_val

RESULTADOS COMPARATIVOS (mAP50-95):
Modelo YOLOv11n: 0.9204


In [14]:
print("\nINFORMACION DEL MODELO YOLOv11n:")
# print(model_final_v11.info)
model_final_v11.info()


INFORMACION DEL MODELO YOLOv11n:
YOLO11n summary (fused): 101 layers, 2,582,347 parameters, 0 gradients, 6.4 GFLOPs


(101, 2582347, 0, 6.3719936)

In [16]:
from torchinfo import summary

def desplegar_arquitectura_completa(modelo_yolo, input_size=(1, 3, 640, 640)):
    """
    Muestra el resumen completo de capas, params y tamaños de memoria.
    input_size: (Batch, Canales, Alto, Ancho)
    """
    print(f"\n🔍 ARQUITECTURA DETALLADA: {modelo_yolo.task_map}")
    # Accedemos al modelo interno de PyTorch (modelo.model)
    summary(modelo_yolo.model, 
            input_size=input_size, 
            col_names=["input_size", "output_size", "num_params", "kernel_size", "mult_adds"],
            verbose=1)

In [18]:
desplegar_arquitectura_completa(model_final_v11, input_size=(1, 3, 640, 640))


🔍 ARQUITECTURA DETALLADA: {'classify': {'model': <class 'ultralytics.nn.tasks.ClassificationModel'>, 'trainer': <class 'ultralytics.models.yolo.classify.train.ClassificationTrainer'>, 'validator': <class 'ultralytics.models.yolo.classify.val.ClassificationValidator'>, 'predictor': <class 'ultralytics.models.yolo.classify.predict.ClassificationPredictor'>}, 'detect': {'model': <class 'ultralytics.nn.tasks.DetectionModel'>, 'trainer': <class 'ultralytics.models.yolo.detect.train.DetectionTrainer'>, 'validator': <class 'ultralytics.models.yolo.detect.val.DetectionValidator'>, 'predictor': <class 'ultralytics.models.yolo.detect.predict.DetectionPredictor'>}, 'segment': {'model': <class 'ultralytics.nn.tasks.SegmentationModel'>, 'trainer': <class 'ultralytics.models.yolo.segment.train.SegmentationTrainer'>, 'validator': <class 'ultralytics.models.yolo.segment.val.SegmentationValidator'>, 'predictor': <class 'ultralytics.models.yolo.segment.predict.SegmentationPredictor'>}, 'pose': {'model'